# New Data Construction 

In [ ]:
# Restart after running this cell.
!pip install -q bitsandbytes accelerate
!python -m spacy download en_core_web_sm -q

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [1]:
import os, gc, re, json, time, pickle, hashlib, urllib.parse, warnings
import numpy as np
import pandas as pd
import torch
import spacy
from tqdm.auto import tqdm
warnings.filterwarnings("ignore", category=FutureWarning)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = "/content/drive/MyDrive/NewDatasetConstruction"
except ImportError:
    BASE_DIR = "./selfcheckgpt_results"
DATA_DIR = os.path.join(BASE_DIR, "new_domain")       # everything for the new-domain study lives here
os.makedirs(DATA_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30 if device.type == "cuda" else 0.0
print(f"Using device: {device}" + (f" - {torch.cuda.get_device_name(0)}" if device.type == "cuda" else "  (WARNING: generation needs a GPU)"))
print("Files are written to", DATA_DIR)
nlp = spacy.load("en_core_web_sm")

def ram(tag=""):
    import psutil
    vm = psutil.virtual_memory(); me = psutil.Process().memory_info().rss / 2**30
    print(f"[RAM {tag}] this process {me:.1f} GB | system used {vm.used / 2**30:.1f} / {vm.total / 2**30:.1f} GB")

Mounted at /content/drive
Using device: cuda - Tesla T4
Files are written to /content/drive/MyDrive/NewDatasetConstruction/new_domain


## Configuration

In [2]:
SEED = 42

GEN_MODEL  = "HuggingFaceH4/zephyr-7b-beta"      # Mistral-7B fine-tune, MIT-licensed, ungated; loaded in 4-bit (as in the pilot)
N_SAMPLES  = 20                                  # paper: 20 stochastic samples per passage (pilot: 3)
MAIN_GREEDY        = True                        # main passage = greedy decoding (SelfCheckGPT); False -> sample it at SAMPLE_TEMPERATURE
SAMPLE_TEMPERATURE = 1.0
SAMPLE_TOP_P       = 1.0                         # 1.0 / 0 = pure sampling as in the paper (the pilot used 0.95 / 50)
SAMPLE_TOP_K       = 0
GEN_4BIT           = "auto"                      # "auto": 4-bit NF4 if GPU memory < 20 GB (T4), fp16 otherwise (L4 / A100: faster); or True / False
MAX_NEW_TOKENS     = 220                         # a 4-6 sentence paragraph is ~120-180 tokens; an unfinished last sentence is trimmed
GEN_BATCH          = 20                          # samples generated per call (halved automatically on OOM)
PROMPT_TEMPLATE    = "Write a short, factual paragraph (4-6 sentences) about {topic}."   # identical to the pilot

MAX_TOPICS_PER_TIER = None                       # None = all 15 per domain and tier (90 topics); e.g. 8 -> 48 topics, 3 -> 18. Ids / seeds do not change, so you can start small and scale up later
MIN_SENTENCES = 3                                # topics whose main passage has fewer sentences are flagged and excluded
MIN_VALID_SAMPLES = 10                           # topics with fewer usable samples are flagged and excluded

np.random.seed(SEED); torch.manual_seed(SEED)

## Topics

In [3]:
TOPICS_RAW = {
 "historical_event": {
  "popular": [
   "the founding of the World Wide Web", "the fall of the Berlin Wall", "the signing of the Magna Carta", "the Apollo 11 moon landing",
   "the sinking of the Titanic", "the storming of the Bastille", "the Battle of Waterloo", "the Russian Revolution of 1917",
   "the assassination of Archduke Franz Ferdinand", "the Boston Tea Party", "the Chernobyl disaster", "the eruption of Mount Vesuvius in 79 AD",
   "the Great Fire of London", "the Treaty of Versailles", "the partition of India in 1947"],
  "less_known": [
   "the Taiping Rebellion", "the Defenestration of Prague in 1618", "the Peterloo Massacre of 1819", "the Battle of Adwa in 1896",
   "the Halifax Explosion of 1917", "the Great Molasses Flood of 1919 in Boston", "the Tunguska event of 1908",
   "the Pig War of 1859 between the United States and Britain", "the Carrington Event solar storm of 1859", "the Treaty of Tordesillas",
   "the Battle of Plassey in 1757", "the Eureka Rebellion of 1854 in Australia", "the Dancing Plague of 1518 in Strasbourg",
   "the Mutiny on the Bounty in 1789", "the Kalinga War of Ashoka"]},
 "company_history": {
  "popular": [
   "the history of the LEGO company", "the history of the Nintendo company", "the history of the Nokia company", "the history of the Toyota company",
   "the history of the IBM company", "the history of the Sony company", "the history of the Coca-Cola company", "the history of the Ford Motor Company",
   "the history of the Nike company", "the history of the Samsung company", "the history of the Kodak company", "the history of the Siemens company",
   "the history of the Volkswagen company", "the history of the Adidas company", "the history of the Microsoft company"],
  "less_known": [
   "the history of the Fiskars company", "the history of the Bang & Olufsen company", "the history of the Tetra Pak company", "the history of the Husqvarna company",
   "the history of the Kikkoman company", "the history of the Faber-Castell company", "the history of the Steiff toy company", "the history of the Bata shoe company",
   "the history of the Marimekko design company", "the history of the Arnott's biscuit company", "the history of the Qantas airline", "the history of the Maersk shipping company",
   "the history of the Carlsberg brewing company", "the history of the Fazer confectionery company", "the history of the Vestas wind turbine company"]},
 "scientific_concept": {
  "popular": [
   "how photosynthesis works", "plate tectonics", "Einstein's theory of general relativity", "DNA replication", "the water cycle",
   "natural selection", "the greenhouse effect", "nuclear fission", "the Krebs cycle", "CRISPR gene editing", "the Big Bang theory",
   "how vaccines produce immunity", "the periodic table of the elements", "how black holes form", "the nitrogen cycle"],
  "less_known": [
   "the Casimir effect", "the Haber-Bosch process", "Bose-Einstein condensates", "the Chandrasekhar limit", "the Mpemba effect",
   "quorum sensing in bacteria", "the Hall effect", "the Meissner effect", "Cherenkov radiation", "the Maillard reaction",
   "the endosymbiotic theory", "the Belousov-Zhabotinsky reaction", "Rayleigh scattering", "the Hardy-Weinberg principle", "the Franck-Condon principle"]},
}
PILOT_TOPICS = {"the founding of the World Wide Web", "the history of the LEGO company", "how photosynthesis works"}
CODE = {"historical_event": "hist", "company_history": "comp", "scientific_concept": "sci"}

TOPICS, n_all = [], 0
for domain, tiers in TOPICS_RAW.items():
    k = 0
    for tier, topics in tiers.items():
        for j, t in enumerate(topics):
            tid, seed_idx = f"{CODE[domain]}_{k:03d}", n_all          # id and seed come from the position in the FULL list ...
            k += 1; n_all += 1
            if MAX_TOPICS_PER_TIER and j >= MAX_TOPICS_PER_TIER: continue   # ... so truncating never changes them
            TOPICS.append({"id": tid, "domain": domain, "tier": tier, "topic": t, "pilot": t in PILOT_TOPICS, "seed_idx": seed_idx})
assert len({t["id"] for t in TOPICS}) == len(TOPICS) and len({t["topic"] for t in TOPICS}) == len(TOPICS)
print(f"{len(TOPICS)} topics")
display(pd.DataFrame(TOPICS).groupby(["domain", "tier"]).size().unstack())

90 topics


tier,less_known,popular
domain,,
company_history,15,15
historical_event,15,15
scientific_concept,15,15


## Generator

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed

gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
USE_4BIT = (GPU_GB < 20) if GEN_4BIT == "auto" else bool(GEN_4BIT)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL, device_map="auto", low_cpu_mem_usage=True,
    **(dict(quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4"))
       if USE_4BIT else dict(torch_dtype=torch.float16)))
print(f"{GEN_MODEL} loaded ({'4-bit NF4' if USE_4BIT else 'fp16'})")
gen_model.eval()
OOM = getattr(torch.cuda, "OutOfMemoryError", RuntimeError)
ram("after loading generator")

def build_prompt(topic):
    msgs = [{"role": "user", "content": PROMPT_TEMPLATE.format(topic=topic)}]
    return gen_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def _generate(prompt, n, sample):
    inputs = gen_tokenizer(prompt, return_tensors="pt").to(gen_model.device)
    kw = dict(max_new_tokens=MAX_NEW_TOKENS, pad_token_id=gen_tokenizer.eos_token_id)
    if sample: kw.update(do_sample=True, temperature=SAMPLE_TEMPERATURE, top_p=SAMPLE_TOP_P, top_k=SAMPLE_TOP_K, num_return_sequences=n)
    else:      kw.update(do_sample=False)
    out = gen_model.generate(**inputs, **kw)
    return gen_tokenizer.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

def generate_many(prompt, n, sample=True):
    outs, todo, cur = [], n, GEN_BATCH
    while todo > 0:
        k = min(cur, todo)
        try:
            outs += _generate(prompt, k, sample)
        except OOM:
            torch.cuda.empty_cache()
            if cur == 1: raise
            cur = max(1, cur // 2); print(f"OOM -> generation batch {cur}"); continue
        todo -= k
    return outs

def clean_passage(text):
    # one paragraph; if generation hit MAX_NEW_TOKENS mid-sentence, cut back to the last complete sentence
    t = re.sub(r"\s+", " ", text).strip()
    if t and t[-1] not in ".!?\"')\u201d\u2019":
        m = re.match(r"^(.*[.!?])(?:\s|$)", t)
        if m: return m.group(1).strip(), True
        return t, True
    return t, False

config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.43k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

... (3 outputs left out)

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

HuggingFaceH4/zephyr-7b-beta loaded (4-bit NF4)
[RAM after loading generator] this process 1.2 GB | system used 4.8 / 12.7 GB


### Generate the dataset

In [ ]:
def cache_path(name): return os.path.join(DATA_DIR, f"cache_{name}.pkl")
def load_cache(name):
    p = cache_path(name)
    if os.path.exists(p):
        with open(p, "rb") as f: return pickle.load(f)
    return {}
def save_cache(name, obj):
    p = cache_path(name); tmp = p + ".tmp"
    with open(tmp, "wb") as f: pickle.dump(obj, f)
    os.replace(tmp, p)

GEN_SIG = hashlib.md5(json.dumps([GEN_MODEL, USE_4BIT, N_SAMPLES, MAIN_GREEDY, SAMPLE_TEMPERATURE, SAMPLE_TOP_P, SAMPLE_TOP_K, MAX_NEW_TOKENS, PROMPT_TEMPLATE, SEED]).encode()).hexdigest()[:8]
GEN_CACHE = f"generations_{GEN_SIG}"

def build_topic(t):
    set_seed(SEED + t["seed_idx"])                                   # per-topic seed: results do not depend on run order / resumption
    prompt = build_prompt(t["topic"])
    main_raw = generate_many(prompt, 1, sample=not MAIN_GREEDY)[0]
    main, main_trim = clean_passage(main_raw)
    samples, n_trim = [], 0
    for s in generate_many(prompt, N_SAMPLES, sample=True):
        s, tr = clean_passage(s); samples.append(s); n_trim += tr
    return {"topic": t["topic"], "main_passage": main, "main_trimmed": main_trim, "samples": samples, "n_samples_trimmed": n_trim}

t0 = time.time()
cache = load_cache(GEN_CACHE)
todo = [t for t in TOPICS if t["id"] not in cache or cache[t["id"]].get("topic") != t["topic"]]   # also regenerates an entry whose topic text differs
print(f"{len(TOPICS) - len(todo)} topics cached, {len(todo)} to generate")
for n, t in enumerate(tqdm(todo, desc="topics")):
    cache[t["id"]] = build_topic(t)
    if (n + 1) % 3 == 0: save_cache(GEN_CACHE, cache)
if todo: save_cache(GEN_CACHE, cache)
print(f"done in {(time.time() - t0) / 60:.1f} min")

first = TOPICS[0]; g = cache[first["id"]]
print(f"\n--- {first['topic']} ---\nMAIN (greedy): {g['main_passage']}\n\nSAMPLE 1: {g['samples'][0]}\n\nSAMPLE 2: {g['samples'][1]}")

0 topics cached, 90 to generate


topics:   0%|          | 0/90 [00:00<?, ?it/s]

done in 69.3 min

--- the founding of the World Wide Web ---
MAIN (greedy): The World Wide Web, the system of interconnected hypertext documents accessed via the Internet, was founded by British computer scientist Tim Berners-Lee in 1989 while working at CERN, the European Organization for Nuclear Research. Berners-Lee proposed a project to create a system for sharing information among researchers, which would allow them to navigate between documents using hypertext links. He developed the first web browser and editor, as well as the first web server, and published a proposal for the web's architecture in 1989. The first web page was created in 1991, and the first web browser, called WorldWideWeb, was released in 1990. The web quickly gained popularity and transformed the way people access and share information.

SAMPLE 1: The World Wide Web, a system of interconnected web pages accessed via the Internet, was founded by British computer scientist Tim Berners-Lee in 1989 while he was wo

## Preprocessing and quality control

In [6]:
REFUSAL = re.compile(r"\b(i'?m sorry|i apologi[sz]e|i cannot|i can'?t|as an ai|i am unable|i'm unable|i do not have|i don'?t have)\b", re.I)
def bad_sample(s): return len(s.split()) < 10 or bool(REFUSAL.search(s))

dataset = []
for t in TOPICS:
    g = cache[t["id"]]
    sentences = [s.text.strip() for s in nlp(g["main_passage"]).sents if s.text.strip()]
    good = [s for s in g["samples"] if not bad_sample(s)]
    flags = []
    if len(sentences) < MIN_SENTENCES: flags.append("short_main")
    if REFUSAL.search(g["main_passage"]): flags.append("refusal_main")
    if len(good) < MIN_VALID_SAMPLES: flags.append("few_samples")
    dataset.append({**t, "prompt": PROMPT_TEMPLATE.format(topic=t["topic"]), "generator": GEN_MODEL,
                    "main_passage": g["main_passage"], "sentences": sentences, "samples": good,
                    "qc": {"n_sentences": len(sentences), "n_samples": len(good), "n_samples_dropped": len(g["samples"]) - len(good),
                           "main_trimmed": bool(g["main_trimmed"]), "n_samples_trimmed": int(g["n_samples_trimmed"]),
                           "n_samples_identical_to_main": sum(s == g["main_passage"] for s in good), "flags": flags, "keep": not flags},
                    "gen_config": {"n_samples": N_SAMPLES, "main_greedy": MAIN_GREEDY, "temperature": SAMPLE_TEMPERATURE, "top_p": SAMPLE_TOP_P,
                                   "top_k": SAMPLE_TOP_K, "max_new_tokens": MAX_NEW_TOKENS, "seed": SEED + t["seed_idx"], "fp16_or_4bit": "4bit" if USE_4BIT else "fp16"}})

qc = pd.DataFrame([{"domain": d["domain"], "tier": d["tier"], "keep": d["qc"]["keep"], "sentences": d["qc"]["n_sentences"],
                    "words": len(d["main_passage"].split()), "samples": d["qc"]["n_samples"], "samples_dropped": d["qc"]["n_samples_dropped"],
                    "main_trimmed": d["qc"]["main_trimmed"], "identical_to_main": d["qc"]["n_samples_identical_to_main"]} for d in dataset])
summary = qc.groupby(["domain", "tier"]).agg(topics=("keep", "size"), kept=("keep", "sum"), avg_sentences=("sentences", "mean"), avg_words=("words", "mean"),
                                             avg_valid_samples=("samples", "mean"), samples_dropped=("samples_dropped", "sum"),
                                             main_trimmed=("main_trimmed", "sum"), samples_identical_to_main=("identical_to_main", "sum")).round(1)
display(summary)
print(f"kept {int(qc.keep.sum())}/{len(qc)} topics, {int(qc[qc.keep].sentences.sum())} sentences to annotate")
flagged = [(d["id"], d["topic"], d["qc"]["flags"]) for d in dataset if not d["qc"]["keep"]]
if flagged: print("flagged:", flagged)
print("\nExample sentences -", dataset[0]["topic"])
for s in dataset[0]["sentences"]: print("  -", s)

topics  kept  avg_sentences  avg_words  \
domain             tier                                                 
company_history    less_known      15    15            4.9      106.6   
                   popular         15    15            5.1      109.7   
historical_event   less_known      15    15            4.6      110.2   
                   popular         15    15            4.4      107.3   
scientific_concept less_known      15    15            4.3      105.9   
                   popular         15    15            4.8      104.0   

                               avg_valid_samples  samples_dropped  \
domain             tier                                             
company_history    less_known               20.0                0   
                   popular                  20.0                0   
historical_event   less_known               20.0                0   
                   popular                  20.0                0   
scientific_concept less_known               20.0                0   
                   popular                  20.0                0   

                               main_trimmed  samples_identical_to_main  
domain             tier                                                 
company_history    less_known             0                          0  
                   popular                0                          0  
historical_event   less_known             0                          0  
                   popular                0                          0  
scientific_concept less_known             0                          0  
                   popular                0                          0

kept 90/90 topics, 421 sentences to annotate

Example sentences - the founding of the World Wide Web
  - The World Wide Web, the system of interconnected hypertext documents accessed via the Internet, was founded by British computer scientist Tim Berners-Lee in 1989 while working at CERN, the European Organization for Nuclear Research.
  - Berners-Lee proposed a project to create a system for sharing information among researchers, which would allow them to navigate between documents using hypertext links.
  - He developed the first web browser and editor, as well as the first web server, and published a proposal for the web's architecture in 1989.
  - The first web page was created in 1991, and the first web browser, called WorldWideWeb, was released in 1990.
  - The web quickly gained popularity and transformed the way people access and share information.


## Export

In [ ]:
# dataset
ds_path = os.path.join(DATA_DIR, "new_domain_dataset.jsonl")
with open(ds_path, "w", encoding="utf-8") as f:
    for d in dataset: f.write(json.dumps(d, ensure_ascii=False) + "\n")

#annotation template (kept topics only)
ann = [{"id": d["id"], "domain": d["domain"], "tier": d["tier"], "topic": d["topic"], "sent_idx": j, "sentence": s, "label": "", "notes": "",
        "wikipedia_search": "https://en.wikipedia.org/w/index.php?search=" + urllib.parse.quote_plus(d["topic"])}
       for d in dataset if d["qc"]["keep"] for j, s in enumerate(d["sentences"])]
tpl_path = os.path.join(DATA_DIR, "new_domain_annotation_TEMPLATE.csv")
pd.DataFrame(ann).to_csv(tpl_path, index=False, encoding="utf-8-sig")

# dataset card
import transformers, sys
card = {"n_topics": len(dataset), "n_topics_kept": int(qc.keep.sum()), "n_sentences_to_annotate": len(ann),
        "domains": sorted({d["domain"] for d in dataset}), "tiers": sorted({d["tier"] for d in dataset}),
        "generator": GEN_MODEL, "generation_signature": GEN_SIG, "config": dataset[0]["gen_config"], "prompt_template": PROMPT_TEMPLATE,
        "versions": {"python": sys.version.split()[0], "torch": torch.__version__, "transformers": transformers.__version__, "spacy": spacy.__version__}}
json.dump(card, open(os.path.join(DATA_DIR, "new_domain_dataset_card.json"), "w"), indent=2)

for p in (ds_path, tpl_path, os.path.join(DATA_DIR, "new_domain_dataset_card.json")): print("wrote", p)
if os.path.exists(os.path.join(DATA_DIR, "new_domain_annotation.csv")): print("\n(note) new_domain_annotation.csv already exists and was NOT touched.")
display(pd.read_csv(tpl_path).head(8)[["id", "tier", "sent_idx", "sentence", "label"]])

wrote /content/drive/MyDrive/NewDatasetConstruction/new_domain/new_domain_dataset.jsonl
wrote /content/drive/MyDrive/NewDatasetConstruction/new_domain/new_domain_annotation_TEMPLATE.csv
wrote /content/drive/MyDrive/NewDatasetConstruction/new_domain/new_domain_dataset_card.json


,id,tier,sent_idx,sentence,label
0,hist_000,popular,0,"The World Wide Web, the system of interconnect...",NaN
1,hist_000,popular,1,Berners-Lee proposed a project to create a sys...,NaN
2,hist_000,popular,2,"He developed the first web browser and editor,...",NaN
3,hist_000,popular,3,"The first web page was created in 1991, and th...",NaN
4,hist_000,popular,4,The web quickly gained popularity and transfor...,NaN
5,hist_001,popular,0,"On November 9, 1989, the Berlin Wall, a physic...",NaN
6,hist_001,popular,1,The event was triggered by a mistake by the Ea...,NaN
7,hist_001,popular,2,"Thousands of people rushed to the wall, chippi...",NaN
